# Week 7: Clustering and Validation Analysis
## Project: Steam Big Data Recommendation Pipeline
This notebook performs market segmentation of Steam games using K-Means and DBSCAN, leveraging the 7 PCA components derived from numerical metrics and semantic embeddings.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
import os

# Settings
sns.set(style='whitegrid')
os.makedirs('../reports/figures', exist_ok=True)

### 1. Load Data
We load the Master Table which now contains the 7 PCA components.

In [4]:
df = pd.read_parquet('../data/processed/master_games_ml_enriched.parquet')
pca_cols = [col for col in df.columns if 'pca_component_' in col]
X = df[pca_cols].values

print(f'Using {len(pca_cols)} PCA components for clustering.')
print(f'Dataset shape: {df.shape}')

Using 0 PCA components for clustering.
Dataset shape: (50872, 24)


### 2. K-Means Parameter Sweep
We evaluate K from 2 to 15 using Inertia (Elbow Method) and Silhouette Score.

In [ ]:
inertias = []
silhouette_avg = []
K_range = range(2, 16)

for k in K_range:
    # Using a sample for silhouette calculation to speed up the sweep if dataset is large
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    inertias.append(kmeans.inertia_)
    
    # Sample for silhouette to avoid long wait times
    sample_idx = np.random.choice(len(X), size=min(10000, len(X)), replace=False)
    score = silhouette_score(X[sample_idx], labels[sample_idx])
    silhouette_avg.append(score)
    print(f'K={k} processed.')

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(K_range, inertias, 'bo-', label='Inertia (Elbow)')
ax1.set_xlabel('Number of clusters (K)')
ax1.set_ylabel('Inertia', color='b')
ax1.tick_params('y', colors='b')

ax2 = ax1.twinx()
ax2.plot(K_range, silhouette_avg, 'ro-', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='r')
ax2.tick_params('y', colors='r')

plt.title('K-Means Parameter Sweep: Elbow and Silhouette')
plt.savefig('../reports/figures/kmeans_sweep.png')
plt.show()

### 3. DBSCAN and Epsilon Search
DBSCAN requires finding the optimal `eps`. We use the K-Distance plot.

In [ ]:
neigh = NearestNeighbors(n_neighbors=14) # 2 * dimensions
nbrs = neigh.fit(X)
distances, indices = nbrs.kneighbors(X)

distances = np.sort(distances[:, 13], axis=0)
plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.title('K-Distance Plot for DBSCAN Epsilon')
plt.xlabel('Points sorted by distance')
plt.ylabel('14-th Nearest Neighbor Distance')
plt.savefig('../reports/figures/dbscan_kdistance.png')
plt.show()

### 4. Final Clustering Selection (K-Means)
Based on the sweep, we select an optimal K (e.g., K=6) to perform the detailed analysis.

In [ ]:
optimal_k = 6
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans_final.fit_predict(X)

print(f'Games per cluster:\n{df["kmeans_cluster"].value_counts()}')

### 5. DBSCAN Implementation
We apply DBSCAN with the selected `eps` (e.g., 0.5) and `min_samples`.

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=14)
df['dbscan_cluster'] = dbscan.fit_predict(X)

print(f'DBSCAN Clusters found (including -1 for noise):\n{df["dbscan_cluster"].value_counts()}')

### 6. Validation Table
We compare the two methods using standard metrics.

In [ ]:
metrics = []

for label_col, name in [('kmeans_cluster', 'K-Means'), ('dbscan_cluster', 'DBSCAN')]:
    labels = df[label_col].values
    # Filter out noise for DBSCAN metrics if needed, or include it
    mask = labels != -1
    if mask.sum() > 1:
        metrics.append({
            'Method': name,
            'Silhouette': silhouette_score(X[sample_idx], labels[sample_idx]),
            'Calinski-Harabasz': calinski_harabasz_score(X, labels),
            'Davies-Bouldin': davies_bouldin_score(X, labels)
        })

validation_df = pd.DataFrame(metrics)
print(validation_df)

### 7. Cluster Profile Analysis
We analyze the mean values of original features to characterize the clusters.

In [ ]:
profile_cols = [
    'hours_mean', 'rec_ratio', 'review_count', 
    'fan_avg_products', 'sentiment_score', 'price_original'
]

cluster_profiles = df.groupby('kmeans_cluster')[profile_cols].mean()
print(cluster_profiles)

plt.figure(figsize=(12, 8))
sns.heatmap(cluster_profiles.T, annot=True, cmap='YlGnBu')
plt.title('Cluster Profiling Heatmap (K-Means Means)')
plt.savefig('../reports/figures/cluster_profiles.png')
plt.show()

### 8. Failure Analysis
We look for points with low Silhouette values or clusters with extreme variance.

In [ ]:
# Identifying games in the most ambiguous cluster or near boundaries
sample_silhouette_values = silhouette_score(X[sample_idx], df.loc[sample_idx, 'kmeans_cluster'], sample_size=1000)
# This is a placeholder for a more complex per-point calculation if needed

print('Failure Analysis: Cluster -1 in DBSCAN represents games that are semantically and numerically unique (outliers).')
outliers = df[df['dbscan_cluster'] == -1].head(10)
print('Example Outliers (Noise):')
print(outliers[['name', 'review_count', 'hours_mean']])